# Sysroot Installation with YAML Configuration

This notebook demonstrates the implementation of sysroot installation using YAML configuration and WSL execution. It includes:

1. Loading and parsing YAML configuration
2. Parsing debootstrap commands
3. Executing commands under WSL
4. Variable substitution
5. Error handling

## 1. Load YAML Configuration

First, we'll import the required libraries and load the YAML configuration file.

In [ ]:
import os
import yaml
import subprocess
import re
from pathlib import Path

def load_yaml_config(config_path):
    try:
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
            return config
    except Exception as e:
        print(f"Error loading YAML configuration: {str(e)}")
        return None

# Load the configuration file
config_path = "settings/sysroot/creation.yaml"
config = load_yaml_config(config_path)

if config:
    print("YAML Configuration loaded successfully:")
    print(yaml.dump(config, default_flow_style=False))
else:
    print("Failed to load configuration")

## 2. Parse Debootstrap Command

Now we'll check if debootstrap commands exist in the configuration and parse them.

In [ ]:
def parse_debootstrap_commands(config):
    commands = []
    if not config or 'sysroot' not in config:
        return commands
    
    for sysroot in config['sysroot']:
        if 'debootstrap_command' in sysroot:
            # Create dictionary of variables to substitute
            variables = {
                'arch': sysroot.get('arch', ''),
                'release': sysroot.get('release', ''),
                'include': ','.join(sysroot.get('include', []))
            }
            commands.append({
                'command': sysroot['debootstrap_command'],
                'variables': variables,
                'sysname': sysroot.get('sysname', ''),
                'version': sysroot.get('version', '')
            })
    return commands

# Parse debootstrap commands from config
commands = parse_debootstrap_commands(config)
print(f"Found {len(commands)} debootstrap commands:")
for cmd in commands:
    print(f"\nSystem: {cmd['sysname']} {cmd['version']}")
    print(f"Command: {cmd['command']}")
    print("Variables:", cmd['variables'])

## 3. Check WSL and Execute Commands

Next, we'll verify that WSL is available and run the debootstrap commands.

In [ ]:
def check_wsl():
    try:
        # Check if WSL is available
        result = subprocess.run(['wsl', '--status'], 
                             capture_output=True, 
                             text=True)
        return result.returncode == 0
    except Exception as e:
        print(f"Error checking WSL: {str(e)}")
        return False

def check_debootstrap():
    try:
        # Check if debootstrap is installed in WSL
        result = subprocess.run(['wsl', 'which', 'debootstrap'],
                              capture_output=True,
                              text=True)
        return result.returncode == 0
    except Exception as e:
        print(f"Error checking debootstrap: {str(e)}")
        return False

# Check WSL and debootstrap availability
wsl_available = check_wsl()
print(f"WSL is {'available' if wsl_available else 'not available'}")

if wsl_available:
    debootstrap_available = check_debootstrap()
    print(f"debootstrap is {'available' if debootstrap_available else 'not available'}")
    
    if not debootstrap_available:
        print("\nTo install debootstrap, run the following in WSL:")
        print("sudo apt-get update && sudo apt-get install -y debootstrap")

## 4. Variable Substitution and Command Execution

Now we'll implement the variable substitution and execute the commands in WSL.

In [ ]:
def substitute_variables(command, variables):
    result = command
    for key, value in variables.items():
        pattern = f"${{{key}}}"
        result = result.replace(pattern, str(value))
    return result

def execute_debootstrap(command_info):
    try:
        # Create the sysroot directory
        sysroot_dir = f"sysroot_{command_info['sysname']}_{command_info['version']}_{command_info['variables']['arch']}"
        sysroot_path = os.path.abspath(sysroot_dir)
        
        # Create directory if it doesn't exist
        os.makedirs(sysroot_path, exist_ok=True)
        
        # Substitute variables in the command
        command = substitute_variables(command_info['command'], command_info['variables'])
        
        # Add the sysroot path to the command
        full_command = f"{command} {sysroot_path}"
        
        # Convert Windows path to WSL path
        wsl_path = subprocess.check_output(['wsl', 'wslpath', '-u', sysroot_path],
                                         text=True).strip()
        
        # Execute the command in WSL
        print(f"\nExecuting debootstrap command for {command_info['sysname']} {command_info['version']}...")
        print(f"Target directory: {sysroot_path}")
        print(f"Command: {full_command}")
        
        result = subprocess.run(['wsl'] + full_command.split(),
                              capture_output=True,
                              text=True)
        
        if result.returncode == 0:
            print("Command executed successfully")
        else:
            print(f"Command failed with error:\n{result.stderr}")
            
        return result.returncode == 0
    
    except Exception as e:
        print(f"Error executing debootstrap: {str(e)}")
        return False

# Execute commands if WSL and debootstrap are available
if wsl_available and debootstrap_available:
    for cmd in commands:
        success = execute_debootstrap(cmd)
        if not success:
            print(f"Failed to create sysroot for {cmd['sysname']} {cmd['version']}")
else:
    print("Cannot execute debootstrap commands: WSL or debootstrap is not available")

## 5. Integration into WorkspaceInstaller Class

Finally, let's implement the install_sysroot method for the WorkspaceInstaller class.